In [1]:
query = """
        WITH CategorizedActivities AS (
    -- Catalog Vists
    SELECT  id_,
        accountid,
        TRUNC(a.CREATEDAT) as activity_day,
        'VISITED_TV_CATALOGUE' as action_type,
        NULL as limit_val

        -- select *
            FROM toki.marketplace_consumer_EVENTS a
    WHERE a.EVENTNAME = 'taxon_click'
      AND JSON_VALUE(a.eventvalue, '$.taxon.label') = N'Зурагт'

    UNION ALL

    -- Product Clicks
    SELECT  id_,
        accountid,
        TRUNC(a.CREATEDAT) as activity_day,
        'VIEWED_TV_PRODUCT' as action_type,
        NULL as limit_val
    FROM toki.marketplace_consumer_EVENTS a
    WHERE a.EVENTNAME = 'product_click'
      AND JSON_VALUE(a.eventvalue, '$.taxon.label') = N'Зурагт'

    UNION ALL

    -- Order Events
    SELECT  distinct a.id_,
        jt.accountId,
        TRUNC(a.CREATEDAT),
        JSON_VALUE(a.ACTIVITYDATA, '$.type'),
        NULL
        -- select *
    FROM toki.marketplace_consumer_activities a,
    JSON_TABLE(
        a.ACTIVITYDATA,
        '$.data'
        COLUMNS (
            accountId VARCHAR2(50) PATH '$.accountId',

            NESTED PATH '$.items[*]'
            COLUMNS (
                productId VARCHAR2(50) PATH '$.productId',
                name VARCHAR2(200) PATH '$.name',
                qty NUMBER PATH '$.qty',
                unitPrice NUMBER PATH '$.unitPrice'
            )
        )
    ) jt


    LEFT JOIN toki.marketplace_catalogue_products p
        ON p.productId = jt.productId
    WHERE a.ACTIVITYNAME = 'order-events' and p_date between '20260201' and '20260228'
      AND JSON_VALUE(p.taxon, '$.label') = N'Зурагт'

    UNION ALL

    -- Wishlist Events
    SELECT  distinct a.id_,
        JSON_VALUE(a.ACTIVITYDATA, '$.accountId'),
        TRUNC(a.CREATEDAT),
        JSON_VALUE(a.ACTIVITYDATA, '$.type'),
        NULL

    -- select *
    FROM toki.marketplace_consumer_activities a
    LEFT JOIN toki.marketplace_catalogue_products p
        ON p.productId = JSON_VALUE(a.ACTIVITYDATA, '$.item')
    WHERE a.ACTIVITYNAME = 'wishlist-events'
    and p_date between '20260201' and '20260228'
      AND JSON_VALUE(p.taxon, '$.label') = N'Зурагт'

    UNION ALL

    -- Cart Events
    SELECT  distinct a.id_,
        jt.accountId,
        TRUNC(a.CREATEDAT),
        JSON_VALUE(a.ACTIVITYDATA, '$.type'),
        NULL

        -- select *
    FROM toki.marketplace_consumer_activities a,
    JSON_TABLE(
        a.ACTIVITYDATA,
        '$.cart'
        COLUMNS (
            accountId VARCHAR2(50) PATH '$.accountId',
            cartid varchar2(50) PATH '$.id_',
            NESTED PATH '$.items[*]'
            COLUMNS (
                productId VARCHAR2(50) PATH '$.productId',
                name VARCHAR2(200) PATH '$.name',
                qty NUMBER PATH '$.qty',
                available NUMBER PATH '$.available'
            )
        )
    ) jt
    LEFT JOIN toki.marketplace_catalogue_products p
        ON p.productId = jt.productId
    WHERE a.ACTIVITYNAME = 'cart-events' and a.p_date between '20260201' and '20260202'
      AND JSON_VALUE(p.taxon, '$.label') = N'Зурагт'

    UNION ALL

    -- Limit Events
    SELECT  distinct a.id_,
        JSON_VALUE(ACTIVITYDATA, '$.accountId'),
        TRUNC(CREATEDAT),
        'LIMIT_CALCULATED',
        JSON_VALUE(ACTIVITYDATA, '$.result.limit')
    FROM toki.marketplace_consumer_activities a
    WHERE ACTIVITYNAME = 'limit-events' and p_date between '20260501' and '20260531'
)
SELECT
    activity_day,
    COUNT(DISTINCT accountid) as UNIQUE_USERS,
    COUNT(CASE WHEN action_type = 'VISITED_TV_CATALOGUE' THEN 1 END) as VISITED_CATALOGUE,
    COUNT(CASE WHEN action_type = 'VIEWED_TV_PRODUCT' THEN 1 END) as VIEWED_PRODUCT,
    COUNT(CASE WHEN action_type = 'ITEM_ADDED' THEN 1 END) as ITEM_ADDED,
    COUNT(CASE WHEN action_type = 'PRODUCT_MODIFIED' THEN 1 END) as CART_MODIFIED,
    COUNT(CASE WHEN action_type = 'PRODUCT_ORDERED' THEN 1 END) as ORDER_ATTEMPTED,
    COUNT(CASE WHEN action_type = 'ORDER_ACTIVATED' THEN 1 END) as ORDER_ACTIVATED,
    COUNT(CASE WHEN action_type = 'LIMIT_CALCULATED' THEN 1 END) as LIMIT_CALCULATED,
    -- Cast limit_val to number for comparison
    COUNT(CASE WHEN action_type = 'LIMIT_CALCULATED' AND TO_NUMBER(limit_val) > 0 THEN 1 END) as LIMIT_GRANTED,
    avg(TO_NUMBER(limit_val)) as avg_limit_amt
FROM CategorizedActivities
WHERE accountid IS NOT NULL
GROUP BY activity_day
ORDER BY activity_day ASC


"""

In [2]:
import os
from src.module.database import oracle_import, oracle_export, oracle_execute
#os.environ

In [3]:
engagement_data = oracle_import(query)

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-07-06 15:40:15.399 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 27 min 


In [4]:
engagement_data.head()

,ACTIVITY_DAY,UNIQUE_USERS,VISITED_CATALOGUE,VIEWED_PRODUCT,ITEM_ADDED,CART_MODIFIED,ORDER_ATTEMPTED,ORDER_ACTIVATED,LIMIT_CALCULATED,LIMIT_GRANTED,AVG_LIMIT_AMT
0,2025-12-30,868,1089,562,0,0,0,0,0,0,NaN
1,2025-12-31,701,844,307,0,0,0,0,0,0,NaN
2,2026-01-01,737,875,395,0,0,0,0,0,0,NaN
3,2026-01-02,968,1211,518,0,0,0,0,0,0,NaN
4,2026-01-03,932,1175,498,0,0,0,0,0,0,NaN
